# pyINLA Interactive Demo

**pyINLA** provides fast, approximate Bayesian inference for latent Gaussian models, directly in Python.

**Why Bayesian inference with INLA?**

- **Full uncertainty quantification**: posterior distributions and credible intervals, not just point estimates
- **Fast**: results in seconds via the INLA approximation, compared to hours with MCMC sampling
- **Hierarchical models**: natural handling of grouped, spatial, and temporal data

This notebook walks through three realistic examples: linear regression, Poisson regression for count data, and hierarchical modeling with random effects. Each example simulates data with known true parameters so you can verify that the model recovers them.

## 1. Setup

Install pyINLA and download the INLA computational binary.

In [ ]:
# Install pyinla
!pip install pyinla -q
print("pyinla installed!")

In [ ]:
# Download the INLA binary (Ubuntu 22.04 for Google Colab)
import pyinla

if not pyinla.is_binary_installed():
    print("Downloading INLA binary for Ubuntu 22.04...")
    pyinla.download_binary(os_name="Ubuntu-22.04", interactive=False)
    print("Done!")
else:
    print("INLA binary already installed")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyinla import pyinla

print("All libraries loaded!")

## 2. Bayesian Linear Regression: Exercise and Heart Rate

**Research question:** Does regular exercise lower resting heart rate?

A health researcher measures resting heart rate (bpm) and weekly exercise hours for 120 adults. Unlike classical regression, the Bayesian approach gives us the full posterior distribution of the exercise effect, directly answering: "How confident are we that exercise lowers heart rate, and by how much?"

**Simulated ground truth:**
- Baseline heart rate (sedentary): 78 bpm
- Effect of exercise: -1.5 bpm per hour of weekly exercise
- Individual variation: SD = 5 bpm

In [ ]:
# Simulate data with known true parameters
np.random.seed(42)

n = 120
exercise_hours = np.random.uniform(0, 12, n)

# True parameters
TRUE_INTERCEPT = 78.0   # Resting HR for someone who doesn't exercise
TRUE_EXERCISE = -1.5    # Each hour of weekly exercise lowers HR by 1.5 bpm
TRUE_NOISE_SD = 5.0

heart_rate = TRUE_INTERCEPT + TRUE_EXERCISE * exercise_hours + np.random.normal(0, TRUE_NOISE_SD, n)

df = pd.DataFrame({'heart_rate': heart_rate, 'exercise': exercise_hours})

print("Sample data (first 5 rows):")
print(df.head())
print(f"\nTrue parameters: intercept = {TRUE_INTERCEPT}, exercise effect = {TRUE_EXERCISE}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['exercise'], df['heart_rate'], alpha=0.6, edgecolors='w', linewidth=0.5)
plt.xlabel('Weekly Exercise (hours)')
plt.ylabel('Resting Heart Rate (bpm)')
plt.title('Does Exercise Lower Resting Heart Rate?')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Define and fit the Bayesian linear regression
model = {
    'response': 'heart_rate',
    'fixed': ['1', 'exercise']  # '1' = intercept
}

result = pyinla(model=model, family='gaussian', data=df)

print("Bayesian Linear Regression Results:")
print(result.summary_fixed)
print(f"\nTrue values: intercept = {TRUE_INTERCEPT}, exercise = {TRUE_EXERCISE}")

In [ ]:
# Visualize the posterior distribution for the exercise effect
marg = result.marginals_fixed['exercise']

plt.figure(figsize=(8, 4))
plt.fill_between(marg['x'], marg['y'], alpha=0.3, color='steelblue')
plt.plot(marg['x'], marg['y'], color='steelblue', linewidth=2, label='Posterior distribution')
plt.axvline(TRUE_EXERCISE, color='red', linestyle='--', linewidth=2, label=f'True value ({TRUE_EXERCISE})')
plt.axvline(0, color='gray', linestyle=':', alpha=0.5, label='No effect')
plt.xlabel('Effect of Exercise on Heart Rate (bpm per hour)')
plt.ylabel('Posterior Density')
plt.title('Posterior Distribution: Exercise Effect')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Print credible interval
est = result.summary_fixed.loc['exercise', :]
print(f"Estimated effect: {est['mean']:.2f} bpm per hour")
print(f"95% credible interval: [{est['0.025quant']:.2f}, {est['0.975quant']:.2f}]")
print(f"\nInterpretation: each additional hour of weekly exercise is associated")
print(f"with a {abs(est['mean']):.1f} bpm reduction in resting heart rate.")

## 3. Poisson Regression: Do Speed Cameras Reduce Accidents?

**Research question:** Do speed cameras reduce traffic accidents at intersections?

A city installs speed cameras at 50 out of 100 high-risk intersections and records accident counts over one year. Since accidents are discrete counts, Poisson regression is the natural model. The Bayesian posterior tells us not just whether cameras help, but quantifies how large the effect is with full uncertainty.

**Simulated ground truth:**
- Baseline accident rate (no camera): exp(1.5) = 4.5 accidents per year
- Camera effect on log-rate: -0.4 (a 33% reduction in accidents)

In [ ]:
np.random.seed(123)

n_intersections = 100
has_camera = np.array([0] * 50 + [1] * 50)  # Balanced: 50 without, 50 with cameras

# True parameters (on the log scale)
TRUE_BASELINE = 1.5     # log(accident rate) without camera
TRUE_CAMERA = -0.4      # log-rate reduction from camera

log_rate = TRUE_BASELINE + TRUE_CAMERA * has_camera
accidents = np.random.poisson(np.exp(log_rate))

df_accidents = pd.DataFrame({
    'accidents': accidents,
    'camera': has_camera
})

print("Accident counts summary:")
print(f"  Without camera: mean = {df_accidents[df_accidents['camera']==0]['accidents'].mean():.1f} per year")
print(f"  With camera:    mean = {df_accidents[df_accidents['camera']==1]['accidents'].mean():.1f} per year")
print(f"\nTrue rates: {np.exp(TRUE_BASELINE):.1f} (no camera), {np.exp(TRUE_BASELINE + TRUE_CAMERA):.1f} (with camera)")

In [ ]:
# Fit Poisson regression
model_poisson = {
    'response': 'accidents',
    'fixed': ['1', 'camera']
}

result_poisson = pyinla(model=model_poisson, family='poisson', data=df_accidents)

print("Poisson Regression Results (log scale):")
print(result_poisson.summary_fixed)
print(f"\nTrue values: intercept = {TRUE_BASELINE}, camera = {TRUE_CAMERA}")

# Interpret on the original scale
est_baseline = result_poisson.summary_fixed.loc['(Intercept)', 'mean']
est_camera = result_poisson.summary_fixed.loc['camera', 'mean']

print(f"\nInterpretation:")
print(f"  Accident rate without camera: exp({est_baseline:.2f}) = {np.exp(est_baseline):.1f} per year")
print(f"  Accident rate with camera:    exp({est_baseline:.2f} + {est_camera:.2f}) = {np.exp(est_baseline + est_camera):.1f} per year")
print(f"  Rate ratio: exp({est_camera:.2f}) = {np.exp(est_camera):.2f}")
print(f"  Cameras are associated with a {(1 - np.exp(est_camera)) * 100:.0f}% reduction in accidents.")

## 4. Hierarchical Model: Student Performance Across Schools

**Research question:** How much does school quality vary, and does study time help equally everywhere?

When data has a grouped structure (students within schools), ignoring group membership can bias estimates. A hierarchical model captures school-level variation through random effects while estimating the overall study-time effect as a fixed effect.

A key benefit of random effects is "shrinkage": extreme school estimates get pulled toward the overall mean. This is especially useful when some schools have few students, because the model borrows strength from other schools.

**Simulated ground truth:**
- Overall intercept: 65 points
- Effect of study hours: 3.0 points per hour
- School-level variation: SD = 8 points
- Individual variation: SD = 5 points

In [ ]:
np.random.seed(456)

n_schools = 10
students_per_school = 25
n_total = n_schools * students_per_school

# True parameters
TRUE_INTERCEPT = 65.0
TRUE_HOURS_EFFECT = 3.0
TRUE_SCHOOL_SD = 8.0
TRUE_NOISE_SD = 5.0

# Generate school-level random effects
true_school_effects = np.random.normal(0, TRUE_SCHOOL_SD, n_schools)

# Generate student data
school_id = np.repeat(range(1, n_schools + 1), students_per_school)
hours = np.random.uniform(1, 8, n_total)
school_effect_per_student = np.array([true_school_effects[s - 1] for s in school_id])

score = (TRUE_INTERCEPT
         + TRUE_HOURS_EFFECT * hours
         + school_effect_per_student
         + np.random.normal(0, TRUE_NOISE_SD, n_total))

df_schools = pd.DataFrame({
    'score': score,
    'hours': hours,
    'school': school_id
})

print(f"Data: {n_total} students across {n_schools} schools")
print(f"\nTrue school effects (deviations from mean):")
for i, eff in enumerate(true_school_effects):
    print(f"  School {i+1}: {eff:+.1f} points")

In [ ]:
plt.figure(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, n_schools))
for i, school in enumerate(range(1, n_schools + 1)):
    mask = df_schools['school'] == school
    plt.scatter(df_schools.loc[mask, 'hours'], df_schools.loc[mask, 'score'],
                color=colors[i], label=f'School {school}', alpha=0.7,
                edgecolors='w', linewidth=0.3)
plt.xlabel('Hours Studied per Week')
plt.ylabel('Exam Score')
plt.title('Exam Scores by School (notice different baseline levels)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Fit hierarchical model with random intercepts per school
model_mixed = {
    'response': 'score',
    'fixed': ['1', 'hours'],
    'random': {
        'school': {'model': 'iid'}
    }
}

result_mixed = pyinla(model=model_mixed, family='gaussian', data=df_schools)

print("Fixed Effects (population-level):")
print(result_mixed.summary_fixed)
print(f"\nTrue values: intercept = {TRUE_INTERCEPT}, hours = {TRUE_HOURS_EFFECT}")

print("\nHyperparameters:")
print(result_mixed.summary_hyperpar)

In [ ]:
# Compare estimated school effects with the true values
estimated_effects = result_mixed.summary_random['school']['mean'].values

plt.figure(figsize=(7, 6))
plt.scatter(true_school_effects, estimated_effects, s=80, zorder=5,
            edgecolors='black', linewidth=0.5, color='steelblue')
for i in range(n_schools):
    plt.annotate(f' {i+1}', (true_school_effects[i], estimated_effects[i]), fontsize=9)

# Reference line for perfect recovery
lims = [min(true_school_effects.min(), estimated_effects.min()) - 2,
        max(true_school_effects.max(), estimated_effects.max()) + 2]
plt.plot(lims, lims, 'k--', alpha=0.3, label='Perfect recovery')

plt.xlabel('True School Effect')
plt.ylabel('Estimated School Effect (posterior mean)')
plt.title('Random Effects: True vs Estimated (shrinkage toward zero)')
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

print("The estimated effects are 'shrunk' toward zero compared to the true values.")
print("This is a feature of Bayesian hierarchical models: it reduces overfitting")
print("by borrowing strength across groups, especially useful when group sizes are small.")

## 5. Try Your Own Data

Upload a CSV file or paste data below, then fit a model.

In [ ]:
from io import StringIO

# Replace this with your own CSV data
my_csv = """
x,y
1,3.2
2,5.1
3,6.8
4,9.3
5,11.0
6,12.5
7,15.1
8,16.8
"""

my_data = pd.read_csv(StringIO(my_csv))
print("Your data:")
print(my_data)

In [ ]:
# Uncomment the lines below to upload a CSV file in Google Colab:

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# my_data = pd.read_csv(filename)
# print(my_data.head())

In [ ]:
# Fit a model to your data
my_model = {
    'response': 'y',       # Change to your response column name
    'fixed': ['1', 'x']    # Change to your predictor column names
}

my_result = pyinla(model=my_model, family='gaussian', data=my_data)
print("Results:")
print(my_result.summary_fixed)

## 6. What's Next?

This demo covered the fundamentals. pyINLA supports much more:

**Likelihood families:**
- `gaussian`: continuous data (shown above)
- `poisson`: count data (shown above)
- `binomial`: binary or proportion data
- `gamma`, `beta`: positive or bounded continuous data
- And 15+ more families

**Random effect models:**
- `iid`: independent random effects (shown above)
- `rw1`, `rw2`: random walks for smooth trends and time series
- `ar1`: autoregressive processes
- `besag`, `bym2`: spatial areal models for disease mapping
- `spde`: continuous spatial fields via stochastic PDEs

### Resources

- **Documentation**: [pyinla.org/docs](https://pyinla.org/docs)
- **Examples**: [pyinla.org/docs/examples](https://pyinla.org/docs/examples)
- **Applications**: [pyinla.org/apps](https://pyinla.org/apps)